# E-Commerce Sales & Customer Behavior Analysis

## 1. Project objective and questions

**Objective:** use the *Online Retail* dataset (a UK-based online gift retailer, transactions from
1 Dec 2010 to 9 Dec 2011) to understand how sales, products, countries and customers behave.

**Questions this notebook answers**

1. How does revenue change month by month?
2. Which products earn the most revenue, and which sell the most units?
3. Which countries generate the most revenue?
4. What does a typical order (invoice) look like in value?
5. How concentrated is revenue among customers?
6. How common are cancellations, and what do they look like?
7. Is there a relationship between unit price and quantity bought?

**How to use it:** run the cells from top to bottom (*Kernel -> Restart & Run All*).
Charts are saved to the `reports/` folder, and the last cells also write the EDA report (PDF) and the PowerPoint.

> **Key definitions (used consistently below)**
> * **Revenue** of a row = `Quantity * UnitPrice` (in GBP).
> * **Cancellation** = an invoice number starting with `C`.
> * **Completed sales** = rows that are not cancellations and have `Quantity > 0` and `UnitPrice > 0`.
> * **Non-product stock codes** = `DOT` (delivery), `POST` (postage), `M` (manual entry).
> * **Product revenue** = completed sales *excluding* the non-product stock codes.

## 2. Importing libraries and loading the Excel data

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# --- Locate the project folders (works whether Jupyter starts in notebooks/ or the project root) ---
cwd = Path.cwd()
PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_DIR / "data" / "Online Retail.xlsx"
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# --- Look and feel ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 150, "axes.titleweight": "bold",
    "axes.titlesize": 13, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#e6e6e6",
})
BLUE, GREY, ORANGE = "#2a6fb0", "#9aa5b1", "#d9822b"
gbp = mtick.FuncFormatter(lambda x, _: f"£{x:,.0f}")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 30

def save_fig(fig, name):
    """Save a figure into reports/ and report where it went."""
    path = REPORTS_DIR / name
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("Saved chart ->", path.relative_to(PROJECT_DIR))

results = {}   # key numbers are collected here and reused by the report / slides at the end

def _fetch_zip(url, dest_dir):
    """Download a zip from UCI and extract it (also unzips zips inside the zip). Returns True on success."""
    import urllib.request, zipfile, io
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        data = urllib.request.urlopen(req, timeout=120).read()
        zf = zipfile.ZipFile(io.BytesIO(data)); zf.extractall(dest_dir)
        for inner in list(Path(dest_dir).glob("*.zip")):
            zipfile.ZipFile(inner).extractall(dest_dir)
        return True
    except Exception as e:
        print("Automatic download failed:", e)
        return False
# --- Get the data: use data/Online Retail.xlsx, or try to download it from UCI if it is missing ---
if not DATA_PATH.exists():
    print("Dataset not found - trying to download it from UCI ...")
    DATA_PATH.parent.mkdir(exist_ok=True)
    _fetch_zip("https://archive.ics.uci.edu/static/public/352/online+retail.zip", DATA_PATH.parent)
if not DATA_PATH.exists():
    raise FileNotFoundError("Please download 'Online Retail.xlsx' from https://archive.ics.uci.edu/dataset/352/online+retail and save it in the data folder.")

# InvoiceNo and StockCode are read as text because some start with letters.
df_raw = pd.read_excel(DATA_PATH, dtype={"InvoiceNo": str, "StockCode": str})
print("Loaded:", DATA_PATH.name)
df_raw.head()

## 3. Initial inspection

`df_raw` is the original data. It is **never modified** - every later step works on copies.

In [ ]:
print(f"Rows: {df_raw.shape[0]:,}   Columns: {df_raw.shape[1]}")
print("\nColumn names:", list(df_raw.columns))
print("\nData types:")
print(df_raw.dtypes)

In [ ]:
missing = pd.DataFrame({
    "missing": df_raw.isna().sum(),
    "percent": (df_raw.isna().mean() * 100).round(2),
})
display(missing)

n_dupes = int(df_raw.duplicated().sum())
print(f"Exact duplicate rows (all 8 columns identical): {n_dupes:,}")
print(f"Date range: {df_raw['InvoiceDate'].min()}  ->  {df_raw['InvoiceDate'].max()}")

results["raw_rows"], results["raw_cols"] = int(df_raw.shape[0]), int(df_raw.shape[1])
results["duplicate_rows"] = n_dupes
results["date_start"] = str(df_raw["InvoiceDate"].min().date())
results["date_end"] = str(df_raw["InvoiceDate"].max().date())

**What to notice:** `CustomerID` has many missing values (guest checkouts / unrecorded customers) and
`Description` has some too. As agreed in the cleaning rules, we **keep** those rows and only drop them
inside an analysis that truly needs the field (e.g. customer rankings need `CustomerID`).

### Data dictionary

| Column | Type | Meaning |
|---|---|---|
| `InvoiceNo` | text | Invoice number (6 digits). A leading **C** marks a cancellation. |
| `StockCode` | text | Product code. `DOT` (delivery), `POST` (postage) and `M` (manual) are not products. |
| `Description` | text | Product name. |
| `Quantity` | integer | Units on the line. Negative on cancellation lines. |
| `InvoiceDate` | datetime | Date and time of the invoice. |
| `UnitPrice` | float | Price per unit in GBP. |
| `CustomerID` | float | 5-digit customer number; missing for unregistered/guest purchases. |
| `Country` | text | Country where the customer lives. |

In [ ]:
display(df_raw[["Quantity", "UnitPrice"]].describe(percentiles=[.01, .25, .5, .75, .99]).T)
display(df_raw[["InvoiceNo", "StockCode", "Description", "CustomerID", "Country"]].astype(str).describe().T)

**Reading the statistics:** `Quantity` and `UnitPrice` both contain negative or zero values (cancellations, adjustments,
free items) and extreme maxima, so they are heavily skewed. We will investigate these in the outlier section.

## 4. Cleaning and analysis datasets

| Dataframe | What it contains | Used for |
|---|---|---|
| `df_raw` | original data, untouched | reference only |
| `df_clean` | `df_raw` minus exact duplicate rows | cancellation analysis |
| `sales` | completed sales (no cancellations, `Quantity > 0`, `UnitPrice > 0`) | monthly revenue |
| `product_sales` | `sales` minus stock codes `DOT`, `POST`, `M` | product, country and order-value analysis |
| `customer_sales` | `sales` rows with a known `CustomerID` | customer analysis |

In [ ]:
NON_PRODUCT_CODES = ["DOT", "POST", "M"]

# 1) Remove exact duplicates into a NEW dataframe (df_raw stays as it was)
df_clean = df_raw.drop_duplicates().copy()

# 2) Helper columns on the cleaned copy
df_clean["StockCode"] = df_clean["StockCode"].str.strip().str.upper()
df_clean["Description"] = df_clean["Description"].str.strip()
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["IsCancellation"] = df_clean["InvoiceNo"].str.startswith("C")
df_clean["Month"] = df_clean["InvoiceDate"].dt.to_period("M")

# 3) Completed sales
sales = df_clean[(~df_clean["IsCancellation"]) & (df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)].copy()

# 4) Product-only sales (drop postage/manual lines)
is_product = ~sales["StockCode"].isin(NON_PRODUCT_CODES)
product_sales = sales[is_product].copy()   # rows with a missing Description are kept; rankings by product drop them automatically

# 5) Known customers
customer_sales = sales[sales["CustomerID"].notna()].copy()
customer_sales["CustomerID"] = customer_sales["CustomerID"].astype(int)

summary = pd.DataFrame({
    "rows": [len(df_raw), len(df_clean), len(sales), len(product_sales), len(customer_sales)],
}, index=["df_raw (original)", "df_clean (no exact duplicates)", "sales (completed)",
          "product_sales (completed, products only)", "customer_sales (completed, known customer)"])
display(summary)

# Revenue totals
total_completed = sales["Revenue"].sum()
total_product = product_sales["Revenue"].sum()
postage_manual = sales.loc[sales["StockCode"].isin(NON_PRODUCT_CODES), "Revenue"].sum()
print(f"Completed-sales revenue (incl. postage/manual): £{total_completed:,.2f}")
print(f"  of which postage/manual lines (DOT, POST, M):  £{postage_manual:,.2f}")
print(f"Product revenue (excl. postage/manual):         £{total_product:,.2f}")

results.update(clean_rows=len(df_clean), sales_rows=len(sales), product_rows=len(product_sales),
               total_completed=float(total_completed), total_product=float(total_product))

## 5. Data quality checks and outliers (univariate view)

Before analysing, we look for **invalid values** and **outliers**. Retail data is naturally right-skewed (many small
purchases, a few huge wholesale ones), so we use the IQR rule only as a *flag* and then investigate - an outlier is not
automatically an error.

**Cleaning decision:** genuine bulk orders are kept (they are real revenue); only rows that fail the completed-sales rules
are excluded.

In [ ]:
print(f"Rows with Quantity <= 0:  {(df_clean['Quantity'] <= 0).sum():,}   (cancellation invoices: {df_clean['IsCancellation'].sum():,})")
print(f"Rows with UnitPrice <= 0: {(df_clean['UnitPrice'] <= 0).sum():,}   (negative prices: {(df_clean['UnitPrice'] < 0).sum():,})")
print(f"Rows with a non-product stock code (DOT/POST/M): {df_clean['StockCode'].isin(NON_PRODUCT_CODES).sum():,}")

odd_codes = df_clean.loc[~df_clean["StockCode"].str.match(r"^\d{5}", na=False), "StockCode"].value_counts().head(12)
print("\nMost common stock codes that do not start with 5 digits (non-standard / non-product):")
display(odd_codes.to_frame("rows"))

# IQR outlier flags on completed product lines
rows = []
for col in ["Quantity", "UnitPrice", "Revenue"]:
    q1, q3 = product_sales[col].quantile([.25, .75]); fence = q3 + 1.5 * (q3 - q1)
    flagged = product_sales[col] > fence
    rows.append({"variable": col, "Q1": q1, "Q3": q3, "upper fence": fence, "lines above fence": int(flagged.sum()),
                 "% of lines": flagged.mean() * 100, "max": product_sales[col].max()})
outlier_tbl = pd.DataFrame(rows).set_index("variable"); display(outlier_tbl)
results["outliers"] = {k: {"fence": float(v["upper fence"]), "n": int(v["lines above fence"]), "pct": float(v["% of lines"])} for k, v in outlier_tbl.iterrows()}

print("\nLargest 10 product lines by revenue:")
display(product_sales.nlargest(10, "Revenue")[["InvoiceNo", "Description", "Quantity", "UnitPrice", "Revenue", "Country"]])
big = product_sales["Quantity"] >= 1000
print(f"Lines with Quantity >= 1,000: {big.sum():,}  ({product_sales.loc[big, 'Revenue'].sum() / total_product * 100:.1f}% of product revenue)")
results["bulk_lines"] = int(big.sum())

**What this tells us:** the IQR rule flags a noticeable share of lines because the data is skewed, not because they are mistakes.
The largest lines are high-quantity bulk purchases (typical of wholesale customers) and are **kept** in the analysis. We
therefore prefer **medians** and **log scales** when summarising and plotting skewed variables.

## 6. Monthly revenue trends

**Definition:** the monthly chart uses **all completed sales, *including* postage/manual lines (DOT, POST, M)**,
because that is the money actually invoiced each month. Because product rankings exclude those lines, the monthly
total is slightly **higher** than the product-revenue total in the cleaning section. For comparison, the grey line shows
product-only revenue.

The data ends part-way through December 2011, so that month is **partial** and is drawn as an open marker.

In [ ]:
monthly = (sales.groupby("Month")["Revenue"].sum().rename("revenue_all_completed").to_frame()
           .join(product_sales.groupby("Month")["Revenue"].sum().rename("revenue_product_only")))
monthly.index = monthly.index.astype(str)
display(monthly)

last_date = df_clean["InvoiceDate"].max()
partial_month = str(last_date.to_period("M"))
# Is the last month incomplete?  (data ends before the last calendar day)
last_is_partial = last_date.normalize() < last_date.to_period("M").end_time.normalize()
print(f"Data ends on {last_date.date()} -> {partial_month} is {'PARTIAL' if last_is_partial else 'complete'}.")

complete = monthly.drop(index=partial_month) if last_is_partial else monthly
low_m, high_m = complete["revenue_all_completed"].idxmin(), complete["revenue_all_completed"].idxmax()
print(f"Lowest month (complete months only):  {low_m}  £{complete.loc[low_m,'revenue_all_completed']:,.2f}")
print(f"Highest month (complete months only): {high_m}  £{complete.loc[high_m,'revenue_all_completed']:,.2f}")
results.update(low_month=low_m, low_month_rev=float(complete.loc[low_m, "revenue_all_completed"]),
               high_month=high_m, high_month_rev=float(complete.loc[high_m, "revenue_all_completed"]),
               partial_month=partial_month if last_is_partial else None, data_end=str(last_date.date()))
results["monthly"] = monthly["revenue_all_completed"].round(2).to_dict()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(monthly))
ax.plot(x, monthly["revenue_all_completed"], color=BLUE, lw=2.2, marker="o", ms=6,
        label="All completed sales (incl. postage/manual)")
ax.plot(x, monthly["revenue_product_only"], color=GREY, lw=1.8, ls="--", label="Products only")
if last_is_partial:   # open marker for the partial month
    ax.plot(x[-1], monthly["revenue_all_completed"].iloc[-1], marker="o", ms=8, mfc="white", mec=BLUE, mew=2)
    ax.annotate("partial month\n(data ends " + last_date.strftime("%d %b") + ")",
                (x[-1], monthly["revenue_all_completed"].iloc[-1]), textcoords="offset points",
                xytext=(-10, 18), ha="right", fontsize=9, color="#444")
for m, lab in [(high_m, "Peak"), (low_m, "Low")]:
    i = list(monthly.index).index(m)
    ax.annotate(f"{lab}: £{monthly.loc[m,'revenue_all_completed']:,.0f}", (i, monthly.loc[m, "revenue_all_completed"]),
                textcoords="offset points", xytext=(0, 10 if lab == "Peak" else -18), ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(monthly.index, rotation=45, ha="right")
ax.yaxis.set_major_formatter(gbp); ax.set_ylim(bottom=0)
ax.set_title("Monthly revenue (completed sales)"); ax.set_xlabel(""); ax.set_ylabel("Revenue")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
save_fig(fig, "01_monthly_revenue.png"); plt.show()

**Reading the chart:** look for seasonal peaks (for a gift retailer, usually in the run-up to Christmas) and for the dip in
the quietest month. The dataset covers just over one year, so we cannot tell how much of any pattern is seasonal versus growth.

In [ ]:
# Weekly revenue (weeks end on Sunday). Drop the first/last bin when the data does not cover a full week.
weekly = sales.set_index("InvoiceDate")["Revenue"].resample("W-SUN").sum()
if sales["InvoiceDate"].min().dayofweek != 0: weekly = weekly.iloc[1:]
if sales["InvoiceDate"].max().dayofweek != 6: weekly = weekly.iloc[:-1]
roll = weekly.rolling(4, center=True).mean()
peak_week = weekly.idxmax()
print(f"{len(weekly)} complete weeks. Peak week ending {peak_week.date()}: £{weekly.max():,.0f}; median week £{weekly.median():,.0f}")
results.update(peak_week=str(peak_week.date()), peak_week_rev=float(weekly.max()), median_week_rev=float(weekly.median()))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(weekly.index, weekly.values, color=GREY, lw=1.4, marker="o", ms=3, label="Weekly revenue")
ax.plot(roll.index, roll.values, color=BLUE, lw=2.6, label="4-week moving average")
ax.annotate(f"Peak week: £{weekly.max():,.0f}", (peak_week, weekly.max()), textcoords="offset points", xytext=(-8, 6), ha="right", fontsize=9)
ax.yaxis.set_major_formatter(gbp); ax.set_ylim(bottom=0)
ax.set_title("Weekly revenue (complete weeks only)"); ax.set_xlabel(""); ax.set_ylabel("Revenue")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
save_fig(fig, "10_weekly_revenue.png"); plt.show()

**Why weekly?** The weekly view smooths day-to-day noise but still reveals short bursts (promotions, pre-holiday ordering)
that the monthly chart hides. The 4-week moving average shows the underlying direction.

## 7. Top products by revenue and by quantity sold

Products are identified by their `Description`. Postage/manual lines are excluded (`product_sales`).

In [ ]:
by_product = product_sales.groupby("Description").agg(
    revenue=("Revenue", "sum"), units=("Quantity", "sum"), invoices=("InvoiceNo", "nunique"))

top_rev = by_product.sort_values("revenue", ascending=False).head(10)
top_qty = by_product.sort_values("units", ascending=False).head(10)
display(top_rev); display(top_qty)

results["top_revenue_product"] = (top_rev.index[0], float(top_rev["revenue"].iloc[0]))
results["top_units_product"] = (top_qty.index[0], int(top_qty["units"].iloc[0]))
results["top_revenue_table"] = {k: round(v, 2) for k, v in top_rev["revenue"].items()}
results["top_units_table"] = {k: int(v) for k, v in top_qty["units"].items()}

def hbar(data, title, fname, color, fmt, xlabel):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    d = data.iloc[::-1]
    ax.barh(d.index.str.title().str.slice(0, 38), d.values, color=color, height=0.68)
    for i, v in enumerate(d.values):
        ax.text(v, i, "  " + fmt(v), va="center", fontsize=9, color="#333")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_xlim(0, d.max() * 1.15)
    ax.grid(axis="y", visible=False)
    save_fig(fig, fname); plt.show()
    return ax

ax = hbar(top_rev["revenue"], "Top 10 products by revenue", "02_top10_products_revenue.png",
          BLUE, lambda v: f"£{v:,.0f}", "Revenue")
ax = hbar(top_qty["units"], "Top 10 products by units sold", "04_top10_products_quantity.png",
          ORANGE, lambda v: f"{v:,.0f}", "Units sold")

In [ ]:
# Sanity check on the quantity ranking: is the #1 product driven by one giant order?
top_name = top_qty.index[0]
rows = product_sales[product_sales["Description"] == top_name]
biggest = rows.nlargest(1, "Quantity").iloc[0]
print(f"'{top_name}': {int(rows['Quantity'].sum()):,} units across {rows['InvoiceNo'].nunique()} invoices.")
print(f"Largest single line: {int(biggest['Quantity']):,} units on invoice {biggest['InvoiceNo']} "
      f"({biggest['Quantity']/rows['Quantity'].sum():.0%} of this product's units).")
results["top_units_single_line_share"] = float(biggest["Quantity"] / rows["Quantity"].sum())

**Caution:** the best-selling product *by units* can be driven by a single bulk order. The cell above shows how
much of its total comes from one invoice, so the unit ranking should be read with that in mind. Revenue and unit
rankings are different lists: cheap items sell in large quantities, while higher-priced items earn more per unit.

### Which products are gaining or losing demand?

For the 30 highest-revenue products we fit a straight line through their **monthly units sold** (complete months only) and
express the slope as *% of the product's average monthly units gained or lost per month*. Positive = rising demand.
This is a simple screening tool: with only about a year of data, seasonal gift items will look like "growth".

In [ ]:
complete_months = [m for m in monthly.index if not (last_is_partial and m == partial_month)]
top30 = by_product["revenue"].nlargest(30).index
pm = product_sales[product_sales["Description"].isin(top30)].copy()
pm["MonthStr"] = pm["Month"].astype(str)
units_m = (pm[pm["MonthStr"].isin(complete_months)]
           .pivot_table(index="MonthStr", columns="Description", values="Quantity", aggfunc="sum", fill_value=0)
           .reindex(complete_months, fill_value=0))
xm = np.arange(len(units_m))
slope = units_m.apply(lambda c: np.polyfit(xm, c.values, 1)[0])
trend_pct = (slope / units_m.mean().replace(0, np.nan) * 100).dropna().sort_values()
growing, declining = trend_pct.tail(5)[::-1], trend_pct.head(5)
display(pd.DataFrame({"product": list(growing.index) + list(declining.index),
                      "trend % per month": list(growing.values) + list(declining.values)}))
results["growing"] = [(k, float(v)) for k, v in growing.items()]
results["declining"] = [(k, float(v)) for k, v in declining.items()]

sel = pd.concat([declining, growing])
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.barh(sel.index.str.title().str.slice(0, 36), sel.values, color=[ORANGE if v < 0 else BLUE for v in sel.values], height=0.7)
ax.axvline(0, color="#444", lw=1); ax.grid(axis="y", visible=False)
ax.set_title("Demand trend of top-30 products: 5 fastest rising and 5 fastest declining")
ax.set_xlabel("Linear trend in monthly units (% of the product's average per month)")
save_fig(fig, "11_product_demand_trend.png"); plt.show()

## 8. Revenue by country

Uses `product_sales` (product revenue, excluding postage/manual lines).

In [ ]:
by_country = product_sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
country_share = (by_country / by_country.sum() * 100)
top_countries = pd.DataFrame({"revenue": by_country.head(10), "share_%": country_share.head(10)})
display(top_countries)
print(f"Countries in the data: {by_country.size}")

uk_rev = float(by_country.get("United Kingdom", 0.0))
results.update(uk_revenue=uk_rev, uk_share=float(uk_rev / by_country.sum() * 100), n_countries=int(by_country.size))
results["top_countries"] = {k: round(v, 2) for k, v in by_country.head(10).items()}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1, 1]})
d = by_country.head(10).iloc[::-1]
axes[0].barh(d.index, d.values, color=BLUE, height=0.68)
for i, v in enumerate(d.values):
    axes[0].text(v, i, f"  £{v:,.0f}", va="center", fontsize=9)
axes[0].set_xlim(0, d.max() * 1.25); axes[0].set_title("Top 10 countries by product revenue")
axes[0].grid(axis="y", visible=False); axes[0].set_xlabel("Revenue")

# The UK dwarfs everyone, so show the rest on their own scale
d2 = by_country.drop("United Kingdom", errors="ignore").head(10).iloc[::-1]
axes[1].barh(d2.index, d2.values, color=GREY, height=0.68)
for i, v in enumerate(d2.values):
    axes[1].text(v, i, f"  £{v:,.0f}", va="center", fontsize=9)
axes[1].set_xlim(0, d2.max() * 1.25); axes[1].set_title("Top 10 excluding the United Kingdom")
axes[1].grid(axis="y", visible=False); axes[1].set_xlabel("Revenue")
fig.tight_layout()
save_fig(fig, "03_top10_countries_revenue.png"); plt.show()

**Reading the chart:** compare the UK bar with the rest; the right-hand panel re-scales the chart (UK removed) so the other
markets can be compared with each other.

## 9. Invoice / order value analysis

**Definition:** the value of an invoice is the sum of `Quantity * UnitPrice` over its **product-only** completed
lines (postage/manual lines and cancellations excluded). Invoices that contain only postage/manual lines are
therefore not counted here.

In [ ]:
invoice_value = product_sales.groupby("InvoiceNo")["Revenue"].sum()
stats = invoice_value.describe(percentiles=[.25, .5, .75, .95, .99])
display(stats.to_frame("invoice value (£)"))

mean_v, median_v = float(invoice_value.mean()), float(invoice_value.median())
results.update(n_invoices=int(invoice_value.size), mean_invoice=mean_v, median_invoice=median_v,
               p99_invoice=float(invoice_value.quantile(.99)), max_invoice=float(invoice_value.max()))
print(f"Product-only invoices: {invoice_value.size:,}   mean £{mean_v:,.2f}   median £{median_v:,.2f}")

cap = invoice_value.quantile(0.99)
shown = invoice_value[invoice_value <= cap]
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(shown, bins=60, color=BLUE, edgecolor="white", ax=ax)
ax.axvline(median_v, color="#222", ls="-", lw=1.5)
ax.axvline(mean_v, color=ORANGE, ls="--", lw=1.8)
ymax = ax.get_ylim()[1]
ax.text(median_v, ymax * 0.95, f" median £{median_v:,.0f}", fontsize=9, va="top")
ax.text(mean_v, ymax * 0.82, f" mean £{mean_v:,.0f}", fontsize=9, va="top", color=ORANGE)
ax.xaxis.set_major_formatter(gbp)
ax.set_title("Order value distribution (product-only invoices)")
ax.set_xlabel(f"Invoice value (shown up to the 99th percentile, £{cap:,.0f}; {(invoice_value > cap).sum()} larger orders hidden)")
ax.set_ylabel("Number of invoices")
save_fig(fig, "05_order_value_distribution.png"); plt.show()

**Reading the chart:** the distribution is strongly right-skewed - most orders are modest, while a few very large
(often wholesale) orders pull the **mean above the median**. The median is therefore the better "typical order" figure.

### Distributions, box plots and order timing (univariate and bivariate views)

The charts below show the **shape** of the main variables (log scales because of the skew), compare order value across
countries and weekdays, and show **when** customers order.

In [ ]:
invoice_df = product_sales.groupby("InvoiceNo").agg(
    value=("Revenue", "sum"), date=("InvoiceDate", "first"), country=("Country", "first"),
    units=("Quantity", "sum"), lines=("StockCode", "count")).reset_index()
invoice_df["weekday"] = invoice_df["date"].dt.day_name()
invoice_df["hour"] = invoice_df["date"].dt.hour
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# ---- 3 histograms (log-spaced bins because the variables are so skewed)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (col, ttl, xl) in zip(axes, [("Quantity", "Units per order line", "Quantity (log scale)"),
                                      ("UnitPrice", "Unit price", "Unit price £ (log scale)"),
                                      ("Revenue", "Revenue per order line", "Line revenue £ (log scale)")]):
    v = product_sales[col]; v = v[v > 0]
    ax.hist(v, bins=np.logspace(np.log10(v.min()), np.log10(v.max()), 45), color=BLUE, edgecolor="white")
    ax.set_xscale("log"); ax.axvline(v.median(), color=ORANGE, ls="--", lw=1.8)
    ax.text(0.97, 0.95, f"median {v.median():,.2f}", transform=ax.transAxes, ha="right", va="top", fontsize=9, color=ORANGE)
    ax.set_title(ttl); ax.set_xlabel(xl); ax.set_ylabel("Number of lines")
fig.tight_layout(); save_fig(fig, "12_distributions.png"); plt.show()

# ---- 3 box plots
top5 = invoice_df["country"].value_counts().head(5).index
product_sales["PriceBand"] = pd.cut(product_sales["UnitPrice"], bins=[0, 1, 2, 5, np.inf], labels=["≤ £1", "£1–2", "£2–5", "> £5"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
sns.boxplot(data=invoice_df[invoice_df["country"].isin(top5)], x="country", y="value", order=top5, color=BLUE, fliersize=2, ax=axes[0])
axes[0].set_yscale("log"); axes[0].set_title("Order value by country (5 busiest)"); axes[0].set_xlabel(""); axes[0].set_ylabel("Invoice value £ (log)")
axes[0].tick_params(axis="x", rotation=30)
present_days = [d for d in DAYS if d in set(invoice_df["weekday"])]
sns.boxplot(data=invoice_df, x="weekday", y="value", order=present_days, color=BLUE, fliersize=2, ax=axes[1])
axes[1].set_yscale("log"); axes[1].set_title("Order value by weekday"); axes[1].set_xlabel(""); axes[1].set_ylabel("Invoice value £ (log)")
axes[1].tick_params(axis="x", rotation=30)
sns.boxplot(data=product_sales, x="PriceBand", y="Quantity", color=BLUE, fliersize=2, ax=axes[2])
axes[2].set_yscale("log"); axes[2].set_title("Units per line by unit-price band"); axes[2].set_xlabel("Unit price band"); axes[2].set_ylabel("Quantity (log)")
fig.tight_layout(); save_fig(fig, "13_boxplots.png"); plt.show()

# ---- 2 count plots: when do customers order?
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=invoice_df, x="weekday", order=DAYS, color=BLUE, ax=axes[0])
axes[0].set_title("Number of orders by weekday"); axes[0].set_xlabel(""); axes[0].set_ylabel("Orders"); axes[0].tick_params(axis="x", rotation=30)
sns.countplot(data=invoice_df, x="hour", color=BLUE, ax=axes[1])
axes[1].set_title("Number of orders by hour of day"); axes[1].set_xlabel("Hour (24h)"); axes[1].set_ylabel("Orders")
fig.tight_layout(); save_fig(fig, "14_order_timing_countplots.png"); plt.show()

# ---- extra: weekday x hour heatmap
heat = invoice_df.pivot_table(index="weekday", columns="hour", values="value", aggfunc="count", fill_value=0).reindex(DAYS).dropna(how="all")
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(heat, cmap="Blues", linewidths=.5, linecolor="white", cbar_kws={"label": "Orders"}, ax=ax)
ax.set_title("When do orders arrive? Orders by weekday and hour"); ax.set_xlabel("Hour of day"); ax.set_ylabel("")
save_fig(fig, "15_weekday_hour_heatmap.png"); plt.show()

by_wd = invoice_df["weekday"].value_counts(); by_hr = invoice_df["hour"].value_counts()
results.update(busiest_weekday=by_wd.index[0], busiest_weekday_orders=int(by_wd.iloc[0]), busiest_hour=int(by_hr.index[0]),
               quietest_weekday=by_wd.index[-1], days_with_orders=int(by_wd.size))
print(f"Busiest weekday: {by_wd.index[0]} ({by_wd.iloc[0]:,} orders); quietest: {by_wd.index[-1]} ({by_wd.iloc[-1]:,}). Busiest hour: {by_hr.index[0]}:00.")
print(f"Median order value by weekday:\n{invoice_df.groupby('weekday')['value'].median().reindex(present_days).round(2).to_string()}")

**How to read these:** in a box plot the box spans the middle 50% of orders and the line is the median; dots are outliers.
Because order values span several orders of magnitude, the log axis makes the boxes comparable. Count plots show *how many*
orders arrive per weekday/hour, which is useful for staffing and for scheduling promotions.

## 10. Customer revenue analysis

Uses `customer_sales`: completed sales with a **known `CustomerID`** (this includes postage/manual lines,
because those charges are billed to the customer). Guest or unrecorded purchases cannot be attributed to a customer.

In [ ]:
cust = customer_sales.groupby("CustomerID").agg(
    revenue=("Revenue", "sum"), orders=("InvoiceNo", "nunique"), units=("Quantity", "sum"))
cust = cust.sort_values("revenue", ascending=False)
share_known = customer_sales["Revenue"].sum() / sales["Revenue"].sum() * 100
print(f"Customers: {len(cust):,}   |   revenue with a known customer: {share_known:.1f}% of completed-sales revenue")
display(cust.head(10))

top10_share = cust["revenue"].head(10).sum() / cust["revenue"].sum() * 100
top20pct_n = int(np.ceil(len(cust) * 0.2))
top20pct_share = cust["revenue"].head(top20pct_n).sum() / cust["revenue"].sum() * 100
repeat_share = (cust["orders"] > 1).mean() * 100
print(f"Top 10 customers = {top10_share:.1f}% of known-customer revenue")
print(f"Top 20% of customers ({top20pct_n:,}) = {top20pct_share:.1f}% of known-customer revenue")
print(f"Customers with more than one order: {repeat_share:.1f}%   |   median orders per customer: {cust['orders'].median():.0f}")

results.update(n_customers=int(len(cust)), known_customer_share=float(share_known),
               top10_cust_share=float(top10_share), top20pct_cust_share=float(top20pct_share),
               repeat_share=float(repeat_share), median_orders=float(cust["orders"].median()),
               top_customer=(int(cust.index[0]), float(cust["revenue"].iloc[0])))

fig, ax = plt.subplots(figsize=(10, 5))
d = cust["revenue"].head(10).iloc[::-1]
ax.barh(d.index.astype(str), d.values, color=BLUE, height=0.68)
for i, v in enumerate(d.values):
    ax.text(v, i, f"  £{v:,.0f}", va="center", fontsize=9)
ax.set_xlim(0, d.max() * 1.15); ax.grid(axis="y", visible=False)
ax.set_title("Top 10 customers by revenue (known CustomerID)"); ax.set_xlabel("Revenue"); ax.set_ylabel("CustomerID")
save_fig(fig, "06_top10_customers_revenue.png"); plt.show()

# Cumulative share curve: how concentrated is revenue?
cum = cust["revenue"].cumsum() / cust["revenue"].sum() * 100
pct_cust = np.arange(1, len(cust) + 1) / len(cust) * 100
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pct_cust, cum.values, color=BLUE, lw=2.2)
ax.plot([0, 100], [0, 100], color=GREY, ls="--", lw=1, label="Perfectly even")
ax.scatter([20], [top20pct_share], color=ORANGE, zorder=3)
ax.annotate(f"Top 20% of customers\n= {top20pct_share:.0f}% of revenue", (20, top20pct_share), xytext=(30, top20pct_share - 18),
            arrowprops=dict(arrowstyle="-", color="#666"), fontsize=10)
ax.set_xlabel("% of customers (largest first)"); ax.set_ylabel("Cumulative % of revenue")
ax.set_title("Customer revenue concentration"); ax.legend(frameon=False, loc="lower right")
save_fig(fig, "07_customer_concentration.png"); plt.show()

In [ ]:
cust_feat = customer_sales.groupby("CustomerID").agg(
    revenue=("Revenue", "sum"), orders=("InvoiceNo", "nunique"), units=("Quantity", "sum"),
    distinct_products=("StockCode", "nunique"), active_months=("Month", "nunique"))
cust_feat["avg_order_value"] = cust_feat["revenue"] / cust_feat["orders"]

rho = float(cust_feat["orders"].corr(cust_feat["revenue"], method="spearman"))
results["orders_revenue_spearman"] = rho
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(cust_feat["orders"], cust_feat["revenue"], s=14, alpha=0.35, color=BLUE, linewidths=0)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Number of orders (log scale)"); ax.set_ylabel("Total revenue £ (log scale)")
ax.set_title("Do frequent customers spend more?"); ax.text(0.04, 0.95, f"Spearman = {rho:.2f}", transform=ax.transAxes, va="top")
save_fig(fig, "16_customer_orders_vs_revenue.png"); plt.show()

corr = cust_feat.corr(method="spearman")
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, linewidths=.5, square=True, ax=ax)
ax.set_title("Customer-level correlation heatmap (Spearman)")
fig.tight_layout(); save_fig(fig, "17_correlation_heatmap.png"); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().sort_values()
print("Strongest relationship:", pairs.index[-1], round(pairs.iloc[-1], 2), "| weakest/most negative:", pairs.index[0], round(pairs.iloc[0], 2))
results["strongest_corr"] = (" vs ".join(pairs.index[-1]), float(pairs.iloc[-1]))

**Multivariate view:** the heatmap shows how customer-level measures move together. Revenue, orders, units and the number of
distinct products tend to rise together (bigger customers do *more of everything*), while average order value is a separate
dimension. Correlation does not prove causation.

## 11. Cancellation analysis

Uses `df_clean` (duplicates removed). An invoice is a **cancellation** if its number starts with `C`.
Cancellation rate = cancelled invoices / all distinct invoices.

In [ ]:
inv_flags = df_clean.groupby("InvoiceNo")["IsCancellation"].first()
n_inv, n_canc = int(inv_flags.size), int(inv_flags.sum())
canc_rate = n_canc / n_inv * 100
print(f"Distinct invoices: {n_inv:,}   cancelled: {n_canc:,}   cancellation rate: {canc_rate:.2f}%")

canc = df_clean[df_clean["IsCancellation"]]
canc_value = -canc["Revenue"].sum()   # cancellation lines have negative quantity
print(f"Cancellation lines: {len(canc):,}   value credited back: £{canc_value:,.2f} "
      f"({canc_value / total_completed * 100:.1f}% of completed-sales revenue)")
print(f"Cancellation lines with Quantity >= 0 (should be none): {(canc['Quantity'] >= 0).sum()}")
results.update(n_invoices_all=n_inv, n_cancelled=n_canc, cancel_rate=float(canc_rate),
               cancel_value=float(canc_value), cancel_value_share=float(canc_value / total_completed * 100))

# Monthly cancellation rate
m = df_clean.groupby(["Month", "InvoiceNo"])["IsCancellation"].first().reset_index()
monthly_c = m.groupby("Month")["IsCancellation"].agg(["sum", "count"])
monthly_c["rate_%"] = monthly_c["sum"] / monthly_c["count"] * 100
monthly_c.index = monthly_c.index.astype(str)

# Most-cancelled products (by units), excluding non-product lines
canc_prod = (canc[~canc["StockCode"].isin(NON_PRODUCT_CODES) & canc["Description"].notna()]
             .groupby("Description")["Quantity"].sum().mul(-1).sort_values(ascending=False).head(10))
display(canc_prod.to_frame("units cancelled"))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(monthly_c.index, monthly_c["rate_%"], color=BLUE, width=0.65)
axes[0].axhline(canc_rate, color=ORANGE, ls="--", lw=1.5)
axes[0].text(0.01, canc_rate, f"overall {canc_rate:.1f}%", color=ORANGE, va="bottom", transform=axes[0].get_yaxis_transform(), fontsize=9)
axes[0].set_title("Share of invoices that are cancellations"); axes[0].tick_params(axis="x", rotation=60)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); axes[0].grid(axis="x", visible=False)
d = canc_prod.iloc[::-1]
axes[1].barh(d.index.str.title().str.slice(0, 34), d.values, color=GREY, height=0.68)
axes[1].set_title("Top cancelled products (units)"); axes[1].grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, "08_cancellations.png"); plt.show()

**Notes:** a cancellation invoice records goods returned or an order reversed; it is not necessarily linked to a
specific earlier invoice in this dataset. Cancellations are counted as a share of *invoices*, not of revenue.

## 12. Relationship between unit price and quantity

Uses `product_sales`. Prices and quantities are heavily skewed (a few huge values), so we use **Spearman rank
correlation**, which only looks at ordering and is not distorted by outliers. We also show the Pearson correlation
of the log-transformed values for comparison.

> **Correlation is not causation.** A negative relationship does *not* prove that lowering prices makes people buy
> more. Cheap items are naturally bought in multiples (e.g. party supplies), and wholesale buyers order cheap items in bulk.

In [ ]:
spearman = float(product_sales["UnitPrice"].corr(product_sales["Quantity"], method="spearman"))
log_pearson = float(np.log10(product_sales["UnitPrice"]).corr(np.log10(product_sales["Quantity"])))
print(f"Spearman correlation (UnitPrice vs Quantity):      {spearman:.3f}")
print(f"Pearson correlation of log10 values:               {log_pearson:.3f}")
a = abs(spearman)
strength = "negligible" if a < 0.1 else "weak" if a < 0.3 else "moderate" if a < 0.5 else "strong"
direction = "negative" if spearman < 0 else "positive"
results["corr_text"] = (f"{strength} {direction} association" +
    ("; higher-priced items tend to be bought in smaller quantities" if spearman <= -0.1 else
     "; higher-priced items tend to be bought in larger quantities" if spearman >= 0.1 else ""))
print("Interpretation:", results["corr_text"])
results.update(spearman=spearman, log_pearson=log_pearson)

# Median quantity per price band
bands = pd.cut(product_sales["UnitPrice"], bins=[0, 1, 2, 3, 5, 10, np.inf],
               labels=["≤ £1", "£1–2", "£2–3", "£3–5", "£5–10", "> £10"])
band_tbl = product_sales.groupby(bands, observed=True)["Quantity"].agg(["median", "mean", "count"])
display(band_tbl)
results["price_band_median_qty"] = {str(k): float(v) for k, v in band_tbl["median"].items()}

sample = product_sales.sample(min(25000, len(product_sales)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(sample["UnitPrice"], sample["Quantity"], s=8, alpha=0.18, color=BLUE, linewidths=0)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Unit price (£, log scale)"); axes[0].set_ylabel("Quantity per line (log scale)")
axes[0].set_title("Unit price vs quantity (random sample)")
axes[0].text(0.97, 0.95, f"Spearman = {spearman:.2f}", transform=axes[0].transAxes, ha="right", va="top", fontsize=11)
axes[1].bar(band_tbl.index.astype(str), band_tbl["median"], color=BLUE, width=0.65)
for i, v in enumerate(band_tbl["median"]):
    axes[1].text(i, v, f"{v:.0f}", ha="center", va="bottom", fontsize=9)
axes[1].set_title("Median quantity per line, by price band"); axes[1].set_xlabel("Unit price band"); axes[1].set_ylabel("Median quantity")
axes[1].grid(axis="x", visible=False)
fig.tight_layout()
save_fig(fig, "09_price_vs_quantity.png"); plt.show()

## 13. Conclusions

### Cross-check against the expected reference results

The table below compares this notebook's calculated numbers with the reference values supplied for the project.
These are **checks, not targets** - nothing in the analysis is forced to match. Any "DIFFERENT" row should be
explained by the definitions above (see the notes printed under the table).

In [ ]:
def chk(label, expected, actual, tol):
    ok = abs(expected - actual) <= tol
    return {"check": label, "expected": expected, "calculated": actual, "result": "match" if ok else "DIFFERENT"}

top_rev_name, top_rev_val = results["top_revenue_product"]
top_units_name, top_units_val = results["top_units_product"]
rows = [
    chk("Original rows", 541909, results["raw_rows"], 0),
    chk("Original columns", 8, results["raw_cols"], 0),
    chk("Exact duplicate rows", 5268, results["duplicate_rows"], 0),
    chk("Rows after removing duplicates", 536641, results["clean_rows"], 0),
    chk("Completed-sales rows", 524878, results["sales_rows"], 0),
    chk("Product revenue total (£)", 10280009.88, results["total_product"], 0.01),
    chk("Lowest month revenue (£)", 522545.56, results["low_month_rev"], 0.01),
    chk("Highest month revenue (£)", 1503866.78, results["high_month_rev"], 0.01),
    chk("Top product revenue (£)", 174156.54, top_rev_val, 0.01),
    chk("Top product units", 80995, top_units_val, 0),
    chk("UK product revenue (£)", 8751658.23, results["uk_revenue"], 0.01),
    chk("UK share of product revenue (%)", 85.13, results["uk_share"], 0.01),
    chk("Product-only invoice count", 19792, results["n_invoices"], 0),
    chk("Average invoice value (£)", 519.40, results["mean_invoice"], 0.01),
    chk("Median invoice value (£)", 302.20, results["median_invoice"], 0.01),
    chk("Distinct invoices (all)", 25900, results["n_invoices_all"], 0),
    chk("Cancelled invoices", 3836, results["n_cancelled"], 0),
    chk("Cancellation rate (%)", 14.81, results["cancel_rate"], 0.01),
    chk("Spearman price-quantity", -0.407, results["spearman"], 0.005),
]
check_df = pd.DataFrame(rows)
display(check_df.style.format({"expected": "{:,.2f}", "calculated": "{:,.2f}"}))
print("Top product by revenue:", top_rev_name, "| expected: REGENCY CAKESTAND 3 TIER")
print("Top product by units:  ", top_units_name, "| expected: PAPER CRAFT , LITTLE BIRDIE")
print("Lowest month:", results["low_month"], "| expected 2011-02;  Highest month:", results["high_month"], "| expected 2011-11")
print("Partial month:", results["partial_month"], "| data ends", results["data_end"])
results["n_checks_match"] = int((check_df["result"] == "match").sum()); results["n_checks"] = len(check_df)

if (check_df["result"] != "match").any():
    print("\nSome values differ from the reference. Likely reasons:")
    print(" - Descriptions are trimmed of leading/trailing spaces here; product rankings group by Description,")
    print("   so spacing/case variants of the same name can split or merge totals.")
    print(" - Stock codes are upper-cased before excluding DOT/POST/M, so one lowercase 'm' (Manual) line worth £2.55 (invoice 539736)")
    print("   is treated as a manual line here; an exact-case match would count it and give £10,280,009.88 instead.")
    print(" - Rows with a missing Description are left out of product-name rankings only (they stay in revenue totals).")
    print(" - Monthly revenue INCLUDES postage/manual lines; product revenue excludes DOT, POST and M, so the two differ.")
    print(" - A different copy/version of the dataset, or a different pandas/Excel-reader version, can change float rounding.")
else:
    print("\nAll checks match the reference results.")

In [ ]:
r = results
print("KEY FINDINGS")
print(f"1. Revenue peaked in {r['high_month']} (£{r['high_month_rev']:,.0f}) and was lowest in {r['low_month']} "
      f"(£{r['low_month_rev']:,.0f}), among complete months. {r['partial_month'] or 'The last month'} is partial (data ends {r['data_end']}).")
print(f"2. Best-selling product by revenue: {r['top_revenue_product'][0]} (£{r['top_revenue_product'][1]:,.0f}); "
      f"by units: {r['top_units_product'][0]} ({r['top_units_product'][1]:,} units, {r['top_units_single_line_share']:.0%} from one order line).")
print(f"3. The United Kingdom provides {r['uk_share']:.1f}% of product revenue (£{r['uk_revenue']:,.0f}); {r['n_countries']} countries in total.")
print(f"4. Typical order: median £{r['median_invoice']:,.0f}, mean £{r['mean_invoice']:,.0f} across {r['n_invoices']:,} product invoices - a few large orders skew the mean upward.")
print(f"5. {r['n_customers']:,} known customers; the top 20% generate {r['top20pct_cust_share']:.0f}% of known-customer revenue; "
      f"{r['repeat_share']:.0f}% ordered more than once.")
print(f"6. {r['n_cancelled']:,} of {r['n_invoices_all']:,} invoices ({r['cancel_rate']:.1f}%) are cancellations.")
print(f"7. Unit price and quantity: Spearman correlation {r['spearman']:.2f} ({r['corr_text']}) - association only, not proof of cause.")

### Insights and business recommendations

The recommendations below are generated from the numbers calculated above, so they always match your run.

In [ ]:
r = results
non_uk = [c for c in by_country.index if c != "United Kingdom"][:3]
recs = [
    f"Plan stock and campaigns around the seasonal peak: revenue was highest in {r['high_month']} (£{r['high_month_rev']:,.0f}) versus a low of £{r['low_month_rev']:,.0f} in {r['low_month']}; build inventory and start promotions in the months before the peak.",
    f"Reduce dependence on the UK ({r['uk_share']:.0f}% of product revenue): test marketing and shipping offers in the strongest overseas markets ({', '.join(non_uk)}).",
    f"Protect high-value customers: the top 20% of known customers generate {r['top20pct_cust_share']:.0f}% of known-customer revenue - introduce account management, loyalty tiers or volume discounts for them.",
    f"Cut cancellations ({r['cancel_rate']:.1f}% of invoices): review the most-cancelled products for quality, description or packaging problems and check the process for wholesale orders.",
    f"Use the typical order (median £{r['median_invoice']:,.0f}) as a benchmark: free-shipping thresholds or bundles slightly above it can lift basket size.",
    (f"Price and quantity show a {r['corr_text']} (Spearman {r['spearman']:.2f}): offer multi-buy/bundle pricing on low-priced items that customers already buy in volume."
     if abs(r['spearman']) >= 0.1 else
     f"Unit price shows only a {r['corr_text']} with quantity (Spearman {r['spearman']:.2f}), so price alone does not explain order size; look at customer type and product category instead."),
    f"Time promotions and support to demand: orders peak on {r['busiest_weekday']} and around {r['busiest_hour']}:00.",
    f"Watch demand trends: {r['growing'][0][0].title()} is rising fastest among top products and {r['declining'][0][0].title()} is declining fastest - confirm with more than one year of data before acting.",
]
results["recommendations"] = recs
for i, t in enumerate(recs, 1):
    print(f"{i}. {t}\n")

### Limitations

* This is **one retailer's historical dataset (Dec 2010 - Dec 2011)**; the results may not generalise to other
  businesses or years, and a single year cannot separate seasonality from growth.
* December 2011 is a **partial month**, so it should not be compared directly with full months.
* About a quarter of rows have no `CustomerID`, so customer results describe only identifiable customers.
* Many customers are wholesalers, so "orders" and "quantities" are not typical consumer behaviour.
* The price-quantity correlation shows association, **not causation**.
* Cancellations are not linked to the original orders, so the cancellation rate is a share of invoices rather than a precise return rate.

## 14. Save results, EDA report (PDF) and presentation (PowerPoint)

The cells below write everything into `reports/`: `results.json`, `EDA_report.pdf`, `final_report.md` and
`presentation.pptx` (8 slides). It needs `reportlab` and `python-pptx` (both are in `requirements.txt`). All numbers come from the calculations above.

In [ ]:
import html
from PIL import Image as PILImage

def build_pdf(path, title, subtitle, sections):
    """sections = [{"heading": str, "paras": [str], "images": [filename in reports/]}]"""
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak, KeepTogether
    ss = getSampleStyleSheet()
    H1 = ParagraphStyle("H1", parent=ss["Title"], fontSize=22, textColor=colors.HexColor("#1b2a41"), spaceAfter=6)
    H2 = ParagraphStyle("H2", parent=ss["Heading2"], textColor=colors.HexColor("#1b2a41"), spaceBefore=10, spaceAfter=4)
    B = ParagraphStyle("B", parent=ss["BodyText"], fontSize=10.5, leading=15, spaceAfter=5)
    SUB = ParagraphStyle("SUB", parent=B, textColor=colors.HexColor("#555555"), alignment=1)
    story = [Spacer(1, 3 * cm), Paragraph(html.escape(title), H1), Paragraph(html.escape(subtitle), SUB), Spacer(1, 1 * cm)]
    for sec in sections:
        block = [Paragraph(html.escape(sec["heading"]), H2)]
        for p in sec.get("paras", []):
            block.append(Paragraph(html.escape(p), B))
        story.append(KeepTogether(block[:2]) if len(block) > 1 else block[0])
        story.extend(block[2:])
        for img in sec.get("images", []):
            fp = REPORTS_DIR / img
            if fp.exists():
                w, h = PILImage.open(fp).size
                width = 16.5 * cm; height = width * h / w
                if height > 11 * cm:
                    height = 11 * cm; width = height * w / h
                story.extend([Image(str(fp), width=width, height=height), Spacer(1, 6)])
    doc = SimpleDocTemplate(str(path), pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm, bottomMargin=1.8 * cm,
                            title=title)
    doc.build(story)

def build_pptx(path, title, subtitle, slides):
    """slides = [{"title": str, "bullets": [str], "images": [filename]}]  (title slide is added first)"""
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.dml.color import RGBColor
    NAVY, TEXT = RGBColor(0x1b, 0x2a, 0x41), RGBColor(0x33, 0x33, 0x33)
    prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)

    def tb(s, x, y, w, h, lines, size, bold=False, color=TEXT):
        box = s.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = box.text_frame; tf.word_wrap = True
        for i, line in enumerate(lines if isinstance(lines, list) else [lines]):
            p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
            p.text = line; p.font.size = Pt(size); p.font.bold = bold; p.font.color.rgb = color; p.space_after = Pt(8)

    def place(s, name, x, y, w, h):
        fp = REPORTS_DIR / name
        if not fp.exists(): return
        iw, ih = PILImage.open(fp).size
        scale = min(w / iw, h / ih); pw, ph = iw * scale, ih * scale
        s.shapes.add_picture(str(fp), Inches(x + (w - pw) / 2), Inches(y), Inches(pw), Inches(ph))

    s = prs.slides.add_slide(prs.slide_layouts[6])
    tb(s, 0.8, 2.0, 11.7, 1.6, title, 38, True, NAVY); tb(s, 0.8, 3.8, 11.7, 1.5, subtitle, 18)
    for sl in slides:
        s = prs.slides.add_slide(prs.slide_layouts[6])
        tb(s, 0.6, 0.3, 12.1, 1.0, sl["title"], 26, True, NAVY)
        imgs, bullets = sl.get("images", []), sl.get("bullets", [])
        if len(imgs) == 1:
            place(s, imgs[0], 0.4, 1.4, 8.2, 5.6); tb(s, 8.8, 1.6, 4.2, 5.4, bullets, 15)
        elif len(imgs) >= 2:
            place(s, imgs[0], 0.3, 1.4, 6.4, 3.9); place(s, imgs[1], 6.7, 1.4, 6.4, 3.9); tb(s, 0.6, 5.5, 12.1, 1.8, bullets, 15)
        else:
            tb(s, 0.8, 1.5, 11.7, 5.5, bullets, 20)
    prs.save(path)

In [ ]:
with open(REPORTS_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

r = results; partial = r["partial_month"]
TITLE = "E-Commerce Sales & Customer Behavior Analysis using Python"
SUB = f"UCI Online Retail dataset | {r['date_start']} to {r['date_end']} | {r['raw_rows']:,} transaction lines | GBP"
growing_txt = ", ".join(f"{k.title()} ({v:+.0f}%/month)" for k, v in r["growing"][:3])
declining_txt = ", ".join(f"{k.title()} ({v:+.0f}%/month)" for k, v in r["declining"][:3])

sections = [
 {"heading": "1. Objective and data", "paras": [
   f"An online gift retailer wants to understand its sales performance, customer purchasing behaviour, product demand and geographic mix. The dataset has {r['raw_rows']:,} transaction lines and {r['raw_cols']} columns covering {r['date_start']} to {r['date_end']}.",
   f"Cleaning: {r['duplicate_rows']:,} exact duplicate rows were removed ({r['clean_rows']:,} remain). Completed sales (not cancelled, quantity and price above zero) total {r['sales_rows']:,} lines. Postage/manual codes (DOT, POST, M) are excluded from product, country and order-value analysis. Rows with a missing CustomerID or Description were kept except where an analysis needs them."]},
 {"heading": "2. Outliers and distributions", "paras": [
   f"Quantity, unit price and line revenue are strongly right-skewed. {r['bulk_lines']:,} lines have a quantity of 1,000 or more; these are genuine bulk orders and were kept, so medians and log scales are used.",
   ], "images": ["12_distributions.png", "13_boxplots.png"]},
 {"heading": "3. Revenue over time", "paras": [
   f"Monthly revenue (all completed sales, including postage/manual lines) peaked in {r['high_month']} at £{r['high_month_rev']:,.0f} and was lowest in {r['low_month']} at £{r['low_month_rev']:,.0f} (complete months only)." + (f" {partial} is a partial month (data ends {r['data_end']})." if partial else ""),
   f"The weekly view peaks in the week ending {r['peak_week']} (£{r['peak_week_rev']:,.0f}) against a median week of £{r['median_week_rev']:,.0f}."],
  "images": ["01_monthly_revenue.png", "10_weekly_revenue.png"]},
 {"heading": "4. Products", "paras": [
   f"Top product by revenue: {r['top_revenue_product'][0].title()} (£{r['top_revenue_product'][1]:,.0f}). Top by units: {r['top_units_product'][0].title()} ({r['top_units_product'][1]:,} units, {r['top_units_single_line_share']:.0%} from one order line, so this ranking is sensitive to bulk orders).",
   f"Fastest-rising top products (linear trend in monthly units): {growing_txt}. Fastest-declining: {declining_txt}. With about one year of data, seasonality can look like a trend."],
  "images": ["02_top10_products_revenue.png", "04_top10_products_quantity.png", "11_product_demand_trend.png"]},
 {"heading": "5. Countries", "paras": [
   f"The United Kingdom contributes £{r['uk_revenue']:,.0f} ({r['uk_share']:.1f}% of product revenue) out of {r['n_countries']} countries."],
  "images": ["03_top10_countries_revenue.png"]},
 {"heading": "6. Orders", "paras": [
   f"There are {r['n_invoices']:,} product-only invoices; the median order is £{r['median_invoice']:,.2f} and the mean £{r['mean_invoice']:,.2f} (a few very large orders pull the mean up). Orders peak on {r['busiest_weekday']} and around {r['busiest_hour']}:00."],
  "images": ["05_order_value_distribution.png", "14_order_timing_countplots.png", "15_weekday_hour_heatmap.png"]},
 {"heading": "7. Customers", "paras": [
   f"{r['n_customers']:,} customers have a known ID (covering {r['known_customer_share']:.0f}% of completed-sales revenue). The top 10 produce {r['top10_cust_share']:.1f}% and the top 20% produce {r['top20pct_cust_share']:.0f}% of known-customer revenue; {r['repeat_share']:.0f}% ordered more than once. Orders and revenue have a Spearman correlation of {r['orders_revenue_spearman']:.2f}."],
  "images": ["06_top10_customers_revenue.png", "07_customer_concentration.png", "16_customer_orders_vs_revenue.png", "17_correlation_heatmap.png"]},
 {"heading": "8. Cancellations", "paras": [
   f"{r['n_cancelled']:,} of {r['n_invoices_all']:,} invoices are cancellations ({r['cancel_rate']:.2f}%), crediting back £{r['cancel_value']:,.0f} ({r['cancel_value_share']:.1f}% of completed-sales revenue)."],
  "images": ["08_cancellations.png"]},
 {"heading": "9. Price and quantity", "paras": [
   f"Spearman correlation between unit price and quantity is {r['spearman']:.3f} ({r['corr_text']}). This is an association only; it does not prove that price changes cause changes in quantity."],
  "images": ["09_price_vs_quantity.png"]},
 {"heading": "10. Recommendations", "paras": r["recommendations"]},
 {"heading": "11. Limitations", "paras": [
   "One retailer's historical data (Dec 2010 - Dec 2011); one year cannot separate seasonality from growth. The last month is partial. About a quarter of rows lack a CustomerID, so customer results cover identifiable customers only. Many customers are wholesalers. Cancellations are not linked to their original orders. Correlations show association, not causation."]},
 {"heading": "12. Cross-check", "paras": [f"{r['n_checks_match']} of {r['n_checks']} supplied reference values matched exactly (see the cross-check table in the notebook)."]},
]
build_pdf(REPORTS_DIR / "EDA_report.pdf", TITLE, SUB, sections)
print("Saved ->", (REPORTS_DIR / "EDA_report.pdf").relative_to(PROJECT_DIR))

# plain-text version of the report
md_txt = f"# {TITLE}\n\n*{SUB}*\n\n" + "\n\n".join(f"## {s['heading']}\n\n" + "\n\n".join(s.get("paras", [])) for s in sections)
(REPORTS_DIR / "final_report.md").write_text(md_txt, encoding="utf-8")

slides = [
 {"title": "Project overview", "bullets": [
   f"{r['raw_rows']:,} transaction lines -> {r['sales_rows']:,} completed-sales lines after cleaning",
   f"Product revenue £{r['total_product']:,.0f} | {r['n_invoices']:,} product invoices | {r['n_customers']:,} known customers",
   "Questions: sales trends, top products, countries, order value, customers, cancellations, price vs quantity",
   "Tools: Python, pandas, NumPy, Matplotlib, Seaborn"]},
 {"title": f"Revenue peaks in {r['high_month']}; weekly view shows the bursts",
  "images": ["01_monthly_revenue.png", "10_weekly_revenue.png"],
  "bullets": [f"Peak month £{r['high_month_rev']:,.0f}; low month £{r['low_month_rev']:,.0f} ({r['low_month']})", (f"{partial} is partial (data ends {r['data_end']})" if partial else "All months complete")]},
 {"title": "Best sellers differ by revenue and by units", "images": ["02_top10_products_revenue.png", "04_top10_products_quantity.png"],
  "bullets": [f"#1 revenue: {r['top_revenue_product'][0].title()} (£{r['top_revenue_product'][1]:,.0f})", f"#1 units: {r['top_units_product'][0].title()} ({r['top_units_product'][1]:,}; {r['top_units_single_line_share']:.0%} from one order line)", f"Rising: {r['growing'][0][0].title()} | Declining: {r['declining'][0][0].title()}"]},
 {"title": f"The UK is {r['uk_share']:.0f}% of revenue; the typical order is £{r['median_invoice']:,.0f}", "images": ["03_top10_countries_revenue.png", "05_order_value_distribution.png"],
  "bullets": [f"Mean order £{r['mean_invoice']:,.0f} > median £{r['median_invoice']:,.0f}: a few big orders skew the average", f"Busiest day: {r['busiest_weekday']}; busiest hour: {r['busiest_hour']}:00"]},
 {"title": "Revenue is concentrated in a minority of customers", "images": ["07_customer_concentration.png", "17_correlation_heatmap.png"],
  "bullets": [f"Top 20% of customers = {r['top20pct_cust_share']:.0f}% of known-customer revenue", f"{r['repeat_share']:.0f}% of customers ordered more than once", f"Orders vs revenue Spearman = {r['orders_revenue_spearman']:.2f}"]},
 {"title": "Cancellations and price vs quantity", "images": ["08_cancellations.png", "09_price_vs_quantity.png"],
  "bullets": [f"{r['cancel_rate']:.1f}% of invoices are cancellations ({r['n_cancelled']:,} of {r['n_invoices_all']:,})", f"Price vs quantity: Spearman {r['spearman']:.2f} ({r['corr_text']}) - association, not causation"]},
 {"title": "Recommendations and limitations", "bullets": [t.split(":")[0] for t in r["recommendations"][:5]] + ["Limitations: one retailer, Dec 2010 - Dec 2011; last month partial; ~25% rows lack CustomerID"]},
]
build_pptx(REPORTS_DIR / "presentation.pptx", TITLE, SUB, slides)
print("Saved ->", (REPORTS_DIR / "presentation.pptx").relative_to(PROJECT_DIR))